# 🧠 Mental Health Prediction – EDA & Training Analysis
**BERT Fine-Tuning for Mental Health Classification**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

plt.style.use('dark_background')
COLORS = ['#4CAF50','#2196F3','#FF9800','#9C27B0','#F44336','#00BCD4','#FF5722']
print('Libraries loaded ✓')

In [ ]:
# Load dataset
df = pd.read_csv('../data/full_dataset.csv')
print(f'Total samples: {len(df)}')
print(df['label_name'].value_counts())

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['label_name'].value_counts()

axes[0].bar(counts.index, counts.values, color=COLORS)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

axes[1].pie(counts.values, labels=counts.index, colors=COLORS, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis
df['text_len'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, group) in enumerate(df.groupby('label_name')):
    axes[0].hist(group['text_len'], bins=20, alpha=0.6, label=label, color=COLORS[i])
    axes[1].hist(group['word_count'], bins=20, alpha=0.6, label=label, color=COLORS[i])

axes[0].set_title('Character Length by Class'); axes[0].legend(fontsize=7)
axes[1].set_title('Word Count by Class'); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Training history (after training)
history_path = Path('../models/bert_mental_health/training_history.json')
if history_path.exists():
    with open(history_path) as f:
        h = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].plot(h['train_loss'], color='#2196F3', label='Train', lw=2)
    axes[0].plot(h['val_loss'], color='#F44336', label='Val', lw=2)
    axes[0].set_title('Loss'); axes[0].legend()

    axes[1].plot(h['train_acc'], color='#4CAF50', label='Train', lw=2)
    axes[1].plot(h['val_acc'], color='#FF9800', label='Val', lw=2)
    axes[1].set_title('Accuracy'); axes[1].legend()

    axes[2].plot(h['val_f1'], color='#9C27B0', lw=2)
    axes[2].set_title('Val F1 Score')

    for ax in axes: ax.grid(alpha=0.3)
    plt.suptitle('BERT Training History', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Train model first: python src/train_bert.py')

In [ ]:
# Model results
results_path = Path('../models/bert_mental_health/results.json')
if results_path.exists():
    with open(results_path) as f:
        r = json.load(f)
    print(f"Test Accuracy : {r['test_accuracy']:.4f}")
    print(f"Test F1 (w)   : {r['test_f1_weighted']:.4f}")

    if 'classification_report' in r:
        report = r['classification_report']
        labels = [k for k in report if k not in ('accuracy','macro avg','weighted avg')]
        f1s = [report[l]['f1-score'] for l in labels]
        plt.figure(figsize=(10, 5))
        bars = plt.bar(labels, f1s, color=COLORS)
        plt.ylim(0, 1)
        plt.title('F1 Score per Class', fontsize=14, fontweight='bold')
        plt.xticks(rotation=45, ha='right')
        for bar, val in zip(bars, f1s):
            plt.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.2f}', ha='center', va='bottom', fontsize=9)
        plt.tight_layout(); plt.show()
else:
    print('Train model first: python src/train_bert.py')